In [ ]:
#read meci, read spawns, align to meci to minimize rmsd. 
#for each spawn, generate all atom type perserving permutations, rotate with scipy.Rotate 

In [ ]:
from pathlib import Path
from scipy.spatial.transform import Rotation as R
import itertools
import numpy as np
import tqdm
import os


In [2]:


def read_xyz_elements(filename):
    with open(filename) as f:
        lines = f.readlines()

    natoms = int(lines[0].strip())
    elems = []

    for line in lines[2:2+natoms]:
        parts = line.split()
        elems.append(parts[0])

    return np.array(elems)


def read_xyz_coordinates(filename):
    with open(filename) as f:
        lines = f.readlines()

    natoms = int(lines[0].strip())
    coords = []

    for line in lines[2:2+natoms]:
        parts = line.split()
        coords.append([float(parts[1]), float(parts[2]), float(parts[3])])

    return np.array(coords, dtype=float)

def generate_type_preserving_permutations(elems):
    groups = {}
    for idx, elem in enumerate(elems):
        groups.setdefault(elem, []).append(idx)

    perm_groups = [list(itertools.permutations(g)) for g in groups.values()]

    all_permutations = []

    for combo in itertools.product(*perm_groups):
        # Start with an identity mapping
        perm = list(range(len(elems)))

        for original_indices, permuted_indices in zip(groups.values(), combo):
            for orig, new in zip(original_indices, permuted_indices):
                perm[orig] = new

        all_permutations.append(perm)

    return all_permutations


def align_and_rmsd_numpy(X, Y, return_aligned=False):
    X_center = X.mean(axis=0)
    Y_center = Y.mean(axis=0)
    Xc = X - X_center
    Yc = Y - Y_center

    rot, _ = R.align_vectors(Xc, Yc)
    Y_aligned = rot.apply(Yc) + X_center
    rmsd = np.sqrt(np.mean(np.sum((X - Y_aligned)**2, axis=1)))

    if return_aligned:
        return rmsd, Y_aligned
    else:
        return rmsd
    

def reflect_axis(geometry, reflection): 
    reflected_geom = geometry * reflection 
    return reflected_geom


REFLECTIONS = np.array(
    [
        [1, 1, 1],
        [-1, 1, 1],
        [1, -1, 1],
        [1, 1, -1],
        [-1, 1, -1],
        [-1, -1, 1],
        [1, -1, -1],
        [-1, -1, -1],
    ]
)

import numpy as np

def write_xyz(filename, elems, coords, comment=""):
  
    coords = np.asarray(coords)

    if coords.shape[0] != len(elems):
        raise ValueError("Number of elements must match number of coordinates.")

    with open(filename, "w") as f:
        f.write(f"{len(elems)}\n")
        f.write(f"{comment}\n")
        for elem, (x, y, z) in zip(elems, coords):
            f.write(f"{elem} {x:.10f} {y:.10f} {z:.10f}\n")


In [ ]:


unaligned_folder = Path('../data/raw_geometries/ethylene/spawn')
meci = Path('../data/raw_geometries/meci/ethylene/0000_2.xyz')

use_reflections = True 

geometries = []
names = []

atoms_meci = read_xyz_elements(meci)
coordinates_meci = read_xyz_coordinates(meci)
permutations = generate_type_preserving_permutations(atoms_meci)

for x in tqdm.tqdm(list(unaligned_folder.glob('*'))):
    coordinates = read_xyz_coordinates(x)
    rmsds = []

    for perm in permutations:
        swapped = coordinates[list(perm)]
        if use_reflections == True: 
            for reflection in REFLECTIONS: 
                swapped_reflection = reflect_axis(swapped, reflection)
                rmsd, aligned_coords = align_and_rmsd_numpy(coordinates_meci, swapped_reflection, return_aligned=True)
                rmsds.append((rmsd, aligned_coords))
        else: 
            rmsd, aligned_coords = align_and_rmsd_numpy(coordinates_meci, swapped, return_aligned=True)
            rmsds.append((rmsd, aligned_coords))

    best_rmsd, best_aligned = min(rmsds, key=lambda t: t[0])
    geometries.append(best_aligned)
    names.append(x.stem)







100%|██████████| 2553/2553 [00:49<00:00, 51.11it/s]


In [ ]:
outdir = Path('../data/aligned_geometries/bruteforce/ethylene')


for i in range(len(geometries)):
    output = outdir / (names[i] + '.xyz')
    #print(output)
    geometry = geometries[i]
    write_xyz(output, atoms_meci, geometry)

In [ ]:

def combine_xyz_files(input_dir, output_file):


    xyz_files = sorted(
        f for f in os.listdir(input_dir)
        if f.lower().endswith(".xyz")
    )

    if not xyz_files:
        raise ValueError("No .xyz files found in the directory.")

    with open(output_file, "w") as outfile:
        for fname in xyz_files:
            full_path = os.path.join(input_dir, fname)

            with open(full_path) as infile:
                lines = infile.readlines()

            # XYZ files: first line is atom count
            i = 0
            while i < len(lines):
                natoms = int(lines[i].strip())
                frame_end = i + 2 + natoms

                # Write this frame into the combined file
                frame = lines[i:frame_end]
                outfile.writelines(frame)

                i = frame_end

    print(f"Combined {len(xyz_files)} files into {output_file}")


In [ ]:
combine_xyz_files(outdir, "../data/aligned_geometries/bruteforce/ethylene_combined_spawns.xyz")


Combined 2663 files into /Users/connerbaucom/Desktop/Pieri/CTG/dim_red_comp/ethylene/rmsd_nograph/alignment/combined.xyz
